# Basic Preprocessing

## Import packages, indicate the /src location, retrieve the data, and prep the corpus for segmentation

In [ ]:
# === Import
import pandas as pd
import re
import sys
import json
from pathlib import Path
import nltk

# === Download NLTK resources if missing ===
try: nltk.data.find("tokenizers/punkt")
except LookupError: nltk.download("punkt")
try: nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    try: nltk.download("punkt_tab")
    except Exception: pass

# === Define the path to the auxiliary modules ===
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not find project root.")

ROOT = find_repo_root()
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
# === Define the path to the data and the pattern for retrieval ==
PROJECT = "identifying_depression_with_rst"
DATA_DIR = (ROOT.parent.parent / "Library" / PROJECT / "data").resolve(strict=True)

# === Pattern ===
data_files_pattern = r"K.+\.csv"

# === Retrieve the data
find_files = DATA_DIR / "raw"

data = []

for item in find_files.iterdir():
   if item.is_file() and re.search(data_files_pattern, item.name):
      data.append((item.stem.lower(), pd.read_csv(item)))


## Coverting the Data into Corpora

In [ ]:
preprocessed_corpora = {
    name: {f"doc-{idx}":{"text": text, "ds": diagnosis}
           for idx, (text, diagnosis) in enumerate(zip(corpus["text"], corpus["group"]))
            }
    for name, corpus in data
                    }


# The structure of the corpora is a nested dictionary:
# {
#     "name_of_corpus_1": {
#         "doc-0": {"text": "Full text of document 0...", "ds": "diagnosis_for_doc_0"},
#         "doc-1": {"text": "Full text of document 1...", "ds": "diagnosis_for_doc_1"},
#         ...
#     },
#     "name_of_corpus_2": {
#         "doc-0": {"text": "...", "ds": "..."},
#         ...
#     }
# }

### (Explicitly) Transform the diagnosis into labels

In [ ]:
# The diagnonses database is structured similar to the corpora
# It's a dictionary with keys for the names of corpora and the diagnoses are lists of string items (as values for the keys)

# Iterate over the database and get only unique labels used
labels = {
    corp: list({doc_cont["ds"] for doc_cont in corp_cont.values()})
    for corp, corp_cont in preprocessed_corpora.items()
}

labels

In [ ]:
map_ = {
    'высокая депрессивность': 1,
    'нет депрессивности': 0,
    'низкая депрессивность': 0,
    'здоровые': 0,
    'депрессия': 1,
    'высокая тревожность': 1,
    'нет тревожности': 0,
    'низкая тревожность': 0
}

for corp_name, corp_data in preprocessed_corpora.items():
    for doc_id, doc_data in corp_data.items():
        diagnosis = doc_data["ds"]
        doc_data["ds_num"] = map_.get(diagnosis, None)

# Saving the corpus/corpora for downstream processing (with an RST parser)

In [ ]:
preprocessed_corpora["ked"]["doc-0"]

In [ ]:
CORPORA = preprocessed_corpora

In [ ]:
save_files_path = DATA_DIR / "interim"
processed_data_file = save_files_path / "preprocesssed_corpora.json"

with open(processed_data_file, "w") as file:
    json.dump(CORPORA, file, indent=4, ensure_ascii=False)